# Building a Local Agent with Strands Agents and Ollama

## Overview

This notebook shows how to build a local agent with Strands Agents and Ollama. The agent runs entirely on your machine and performs local file operations: reading files (including PDFs), writing files, and listing directory contents.

## What is Ollama?

[Ollama](https://ollama.com/) is an open-source framework for running large language models (LLMs) locally on your own machine. It exposes a simple local API, which makes it a good fit for privacy-focused or offline use cases where you do not want to send data to an external service.

Key benefits of Ollama:
- **Privacy**: All processing happens locally on your machine
- **No API costs**: Free to use as much as you want
- **Offline capability**: Works without an internet connection
- **Customization**: Can be fine-tuned for specific use cases

## Tutorial Details

| Information            | Details                                        |
|:-----------------------|:-----------------------------------------------|
| Agent structure        | Single agent                                   |
| Model                  | `qwen2.5:1.5b` (any tool-capable model you pull with Ollama) |
| Runs on                | Your local machine or a CPU-only cloud environment (via Ollama) |
| Strands model provider | `OllamaModel`                                   |
| Custom tools           | file_read, file_write, list_directory          |

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## What you'll learn
* Configure a local model with the `OllamaModel` provider
* Give an agent custom tools with the `@tool` decorator
* Build a file-operations agent (read, write, list) that runs entirely on your machine

In [ ]:
!pip install -q -r requirements.txt

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* Ollama ([download](https://ollama.com/download)): the cells below install and start it for a Linux/Amazon SageMaker Studio environment; install it yourself if you are running locally

Before running this notebook, make sure you have:

1. Installed Ollama: [https://ollama.com/download](https://ollama.com/download)
2. Started the Ollama server: `ollama serve`
3. Downloaded a model with Ollama: `ollama pull qwen2.5:1.5b`

Refer to the [Ollama model provider documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/ollama/) for detailed instructions.

In this notebook, we install Ollama for the Linux distribution for compatibility with Amazon SageMaker Studio. This is done for code execution during AWS-led workshops on AWS Workshop Studio. If you are running this code locally, adjust the steps to install Ollama for your own environment.

In [ ]:
# this will work on linux computers
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess

# Start the Ollama server in the background. Redirect its output so the
# server logs do not fill the notebook
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

In [ ]:
!ollama pull qwen2.5:1.5b

In [ ]:
# Import necessary libraries
import os

import nest_asyncio
import requests

# Import strands components
from strands import Agent, tool
from strands.models.ollama import OllamaModel

# Allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

In [ ]:
# Check if Ollama is running:
try:
    response = requests.get("http://localhost:11434/api/tags")
    print("✅ Ollama is running. Available models:")
    for model in response.json().get("models", []):
        print(f"- {model['name']}")
except requests.exceptions.ConnectionError:
    print("❌ Ollama is not running. Please start Ollama before proceeding.")

## Defining custom tools

Tools are functions that the agent can use to interact with the environment. Below, we define several tools for our file-operations agent.

In [ ]:
# File Operation Tools
@tool
def file_read(file_path: str) -> str:
    """Read a file and return its content. Supports both text and PDF files.

    Args:
        file_path (str): Path to the file to read

    Returns:
        str: Content of the file

    Raises:
        FileNotFoundError: If the file doesn't exist
    """
    try:
        # Check if it's a PDF file
        if file_path.lower().endswith('.pdf'):
            import PyPDF2
            with open(file_path, "rb") as file:
                pdf_reader = PyPDF2.PdfReader(file)
                text = ""
                for page in pdf_reader.pages:
                    text += page.extract_text() + "\n"
                return text if text.strip() else "Error: Could not extract text from PDF"
        else:
            # Regular text file
            with open(file_path, "r", encoding="utf-8") as file:
                return file.read()
    except FileNotFoundError:
        return f"Error: File '{file_path}' not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"


@tool
def file_write(file_path: str, content: str) -> str:
    """Write content to a file.

    Args:
        file_path (str): The path to the file
        content (str): The content to write to the file

    Returns:
        str: A message indicating success or failure
    """
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(os.path.abspath(file_path)), exist_ok=True)

        with open(file_path, "w") as file:
            file.write(content)
        return f"File '{file_path}' written successfully."
    except Exception as e:
        return f"Error writing to file: {str(e)}"


@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories in the specified path.

    Args:
        directory_path (str): Path to the directory to list

    Returns:
        str: A formatted string listing all files and directories
    """
    try:
        items = os.listdir(directory_path)
        files = []
        directories = []

        for item in items:
            full_path = os.path.join(directory_path, item)
            if os.path.isdir(full_path):
                directories.append(f"Folder: {item}/")
            else:
                files.append(f"File: {item}")

        result = f"Contents of {os.path.abspath(directory_path)}:\n"
        result += (
            "\nDirectories:\n" + "\n".join(sorted(directories))
            if directories
            else "\nNo directories found."
        )
        result += (
            "\n\nFiles:\n" + "\n".join(sorted(files)) if files else "\nNo files found."
        )

        return result
    except Exception as e:
        return f"Error listing directory: {str(e)}"

## Creating the Ollama-powered agent

Now we will create our agent using the Ollama model and the tools we defined above.

Note: You can add more tools like `execute_commands`, `search_files` etc.

In [ ]:
# Define a comprehensive system prompt for our agent
system_prompt = """
You are a helpful personal assistant capable of performing local file actions and simple tasks for the user.

Your key capabilities:
1. Read, understand, and summarize files.
2. Create and write to files.
3. List directory contents and provide information on the files.
4. Summarize text content

When using tools:
- Always verify file paths before operations
- Be careful with system commands
- Provide clear explanations of what you're doing
- If a task cannot be completed, explain why and suggest alternatives

Always be helpful, concise, and focus on addressing the user's needs efficiently.
"""

model_id = (
    "qwen2.5:1.5b"  # You can change this to any tool-capable model you have pulled with Ollama.
)

### Configure the Ollama model
Make sure the Ollama service is running at http://localhost:11434 and your `model_id` appears in the list of models printed above.

This notebook uses `qwen2.5:1.5b`, which supports tool calling. The agent needs that to invoke the file tools. Any tool-capable Ollama model works; a recent Ollama version is recommended so it can pull current models.

In [ ]:
ollama_model = OllamaModel(
    model_id=model_id,
    host="http://localhost:11434",
    max_tokens=4096,  # Maximum tokens to generate in the response
    temperature=0.7,  # Lower for more deterministic responses, higher for more creative
    top_p=0.9,  # Nucleus sampling parameter
    options={"num_ctx": 16384},  # Input context window; widen it so long inputs are not truncated
)

# Create the agent
local_agent = Agent(
    system_prompt=system_prompt,
    model=ollama_model,
    tools=[file_read, file_write, list_directory],
)

## Testing the agent

Let's test our agent with some example tasks.

In [ ]:
local_agent(
    "Read the file in the path `sample_file/Amazon-com-Inc-2023-Shareholder-Letter.pdf` and summarize it in 5 bullet points."
)

In [ ]:
# Example 2: List files in the current directory
response = local_agent("Show me the files in the current directory")

In [ ]:
# Example 3: Create a sample file
response = local_agent(
    "Create a file called 'sample.txt' with the content 'This is a test file created by my Ollama agent.'"
)

In [ ]:
# Example 4: create a readme file after reading and understanding multiple files
response = local_agent("Create a readme.md for the current directory")

## Summary

In this notebook you built a local agent with Strands Agents and Ollama. The agent runs entirely on your machine and can read files (including PDFs), write files, and list directory contents through custom tools.

This shows how running a model locally with Ollama combines with the flexibility of the Strands Agents tool system. You can extend this agent by adding more tools or switching to a different Ollama model. Next, let's look at reaching a hosted model through LiteLLM.